Importy

In [ ]:
import pickle
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
import csv
from tqdm import tqdm

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Używane urządzenie: {device}")

Używane urządzenie: cuda


Wczytanie danych

In [8]:
with open('/content/train.pkl', 'rb') as f:
    train_data = pickle.load(f)

with open('/content/test_no_target.pkl', 'rb') as f:
    test_data = pickle.load(f)

import random
random.seed(42)

from collections import defaultdict
by_label = defaultdict(list)
for seq, label in train_data:
    by_label[label].append((seq, label))

train_data, val_data = [], []
for label, samples in by_label.items():
    random.shuffle(samples)
    split = int(0.15 * len(samples))
    val_data.extend(samples[:split])
    train_data.extend(samples[split:])

random.shuffle(train_data)
random.shuffle(val_data)
print(f"Dane treningowe: {len(train_data)}, walidacyjne: {len(val_data)}")

vocab = set()
for seq, _ in train_data:
    vocab.update(seq)

word2idx = {chord: i+1 for i, chord in enumerate(vocab)}
word2idx['<PAD>'] = 0
vocab_size = len(word2idx)

print(f"Rozmiar słownika (akordy z trainu + PAD): {vocab_size}")


Dane treningowe: 2500, walidacyjne: 439
Rozmiar słownika (akordy z trainu + PAD): 183


Wczytanie datasetu

In [9]:
class ChordDataset(Dataset):
    def __init__(self, data, word2idx, is_test=False):
        self.is_test = is_test
        if not is_test:
            self.seqs = [torch.tensor([word2idx.get(w, 0) for w in seq], dtype=torch.long) for seq, label in data]
            self.labels = [label for seq, label in data]
        else:
            self.seqs = [torch.tensor([word2idx.get(w, 0) for w in seq], dtype=torch.long) for seq in data]

    def __len__(self):
        return len(self.seqs)

    def __getitem__(self, idx):
        if not self.is_test:
            return self.seqs[idx], self.labels[idx]
        return self.seqs[idx]

def collate_train(batch):
    seqs, labels = zip(*batch)
    lengths = torch.tensor([len(s) for s in seqs], dtype=torch.long)
    seqs_padded = pad_sequence(seqs, batch_first=True, padding_value=word2idx['<PAD>'])
    return seqs_padded, torch.tensor(labels, dtype=torch.long), lengths

def collate_test(batch):
    seqs = batch
    lengths = torch.tensor([len(s) for s in seqs], dtype=torch.long)
    seqs_padded = pad_sequence(seqs, batch_first=True, padding_value=word2idx['<PAD>'])
    return seqs_padded, lengths

BATCH_SIZE = 64

train_dataset = ChordDataset(train_data, word2idx, is_test=False)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_train)

val_dataset = ChordDataset(val_data, word2idx, is_test=False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_train)

test_dataset = ChordDataset(test_data, word2idx, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_test)

Definicja modelu

In [10]:
class ComposerRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super(ComposerRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=word2idx['<PAD>'])
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=2, batch_first=True, dropout=0.3)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x, lengths):
        embedded = self.embedding(x)
        packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (hidden, _) = self.lstm(packed)
        out = self.fc(hidden[-1])
        return out

EMBED_DIM = 64
HIDDEN_DIM = 128
NUM_CLASSES = 5

model = ComposerRNN(vocab_size, EMBED_DIM, HIDDEN_DIM, NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)

Uczenie

In [11]:
EPOCHS = 40
PATIENCE = 8

best_val_acc = 0.0
epochs_no_improve = 0
best_model_state = None

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for seqs_padded, labels, lengths in tqdm(train_loader, desc=f"Epoka {epoch+1}/{EPOCHS}"):
        seqs_padded, labels = seqs_padded.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(seqs_padded, lengths)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = 100 * correct / total
    train_loss = total_loss / len(train_loader)

    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for seqs_padded, labels, lengths in val_loader:
            seqs_padded, labels = seqs_padded.to(device), labels.to(device)
            outputs = model(seqs_padded, lengths)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_loss /= len(val_loader)
    val_acc = 100 * val_correct / val_total

    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoka {epoch+1}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}% | "
          f"LR: {current_lr:.6f}")

    scheduler.step(val_loss)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"\nEarly stopping po epoce {epoch+1} (brak poprawy przez {PATIENCE} epok).")
            break

model.load_state_dict(best_model_state)
print(f"\nZaładowano najlepszy model (val acc: {best_val_acc:.2f}%)")


Epoka 1/40: 100%|██████████| 40/40 [00:12<00:00,  3.33it/s]


Epoka 1/40 | Train Loss: 1.2269 | Train Acc: 56.20% | Val Loss: 1.0433 | Val Acc: 62.41% | LR: 0.001000


Epoka 2/40: 100%|██████████| 40/40 [00:09<00:00,  4.34it/s]


Epoka 2/40 | Train Loss: 0.9279 | Train Acc: 64.52% | Val Loss: 0.8478 | Val Acc: 71.75% | LR: 0.001000


Epoka 3/40: 100%|██████████| 40/40 [00:09<00:00,  4.43it/s]


...

Epoka 20/40 | Train Loss: 0.1256 | Train Acc: 97.04% | Val Loss: 0.3647 | Val Acc: 89.52% | LR: 0.000500


Epoka 21/40: 100%|██████████| 40/40 [00:08<00:00,  4.78it/s]


Epoka 21/40 | Train Loss: 0.1368 | Train Acc: 95.32% | Val Loss: 0.3511 | Val Acc: 89.07% | LR: 0.000500


Epoka 22/40: 100%|██████████| 40/40 [00:08<00:00,  4.51it/s]


Epoka 22/40 | Train Loss: 0.0833 | Train Acc: 97.56% | Val Loss: 0.3469 | Val Acc: 89.98% | LR: 0.000500

Early stopping po epoce 22 (brak poprawy przez 8 epok).

Załadowano najlepszy model (val acc: 90.66%)


Predykcje

In [13]:
model.eval()
all_predictions = []

with torch.no_grad():
    for seqs_padded, lengths in tqdm(test_loader, desc="Przewidywanie na zbiorze testowym"):
        seqs_padded = seqs_padded.to(device)
        outputs = model(seqs_padded, lengths)
        _, predicted = torch.max(outputs.data, 1)
        all_predictions.extend(predicted.cpu().numpy())

with open('pred.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    for pred in all_predictions:
        writer.writerow([pred])

Przewidywanie na zbiorze testowym: 100%|██████████| 18/18 [00:02<00:00,  7.54it/s]
